#### 문장의 유사도 분석
: 두개의 문장이 비슷한 것인지 또는 관련이 있는것인지 분석

#### 코사인(Cosine) 유사도

In [27]:
a="딥러닝은 매우 재미있는 기술이라 공부하고 있습니다."
# b="공부하면 재미있는 기술이라 딥러닝을 배우고 있습니다."
b="동해물과 백두산이 마르고 닳도록 하느님이 보우하사 우리나라만세."

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [29]:
sentence = (a,b)
tfid_vectorizer = TfidfVectorizer()

In [30]:
sentence

('딥러닝은 매우 재미있는 기술이라 공부하고 있습니다.', '동해물과 백두산이 마르고 닳도록 하느님이 보우하사 우리나라만세.')

In [31]:
# 문장 벡터화 하기 : 사전 만들기
tfid_matrix = tfid_vectorizer.fit_transform(sentence)

In [32]:
from sklearn.metrics.pairwise import cosine_similarity
# 첫번째 문장과 두번쨰 문장 비교
cos_smilar=cosine_similarity(tfid_matrix[0:1], tfid_matrix[1:2])
print("코사인 유사도 측정:", cos_smilar)

코사인 유사도 측정: [[0.]]


----
#### 유사도를 이용한 추천 시스템 구현
: 코사인 유사도만으로 영화의 줄거리에 ...

In [33]:
import pandas as pd

In [34]:
data = pd.read_csv("../Data/movies_metadata.csv")

/var/folders/y7/cnx_3pbd4vg17lq55z4_mk7w0000gn/T/ipykernel_3073/859470591.py:1: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("../Data/movies_metadata.csv")


In [35]:
data.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [36]:
data[['title', 'overview']].head()

,title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...


In [37]:
# 상위 2만개
data = data.head(20000)

In [38]:
data['overview'].isnull().sum()

135

In [39]:
# 결측치를 반값으로 대체
data['overview'] = data['overview'].fillna(" ")

In [40]:
# 행렬 크기 구하기
tfid = TfidfVectorizer(stop_words='english')
tfid_matrix = tfid.fit_transform(data['overview'])
tfid_matrix.shape

# 결과 (20000,47487)

(20000, 47487)

In [41]:
cosine_sim = cosine_similarity(tfid_matrix, tfid_matrix)

In [42]:
# 기존 데이터프레임으로 부터 영화 타이틀을 key, 영화 index를 value하는 딕셔너리
title_to_index = dict(zip(data['title'],data.index ))

In [43]:
title_to_index['Father of the Bride Part II']

4

In [44]:
# 선택한 영화의 제목을 입력하면 코사인 유사도를 통해 가장 overview가
# 유사한 10개의 영화를 찾아내는 함수

def get_recommendation(title,cosine_sim=cosine_sim):
   # 선택한 영화의 타이틀로부터 해당 영화의 인덱스를 받아온다.
   idx = title_to_index[title]
   # 해당 영화와 모든 영화와의 유사도를 가져온다.
   sim_scores = list(enumerate(cosine_sim[idx]))
   # 유사도에 따라 영화들을 정렬한다.
   sim_scores = sorted(sim_scores,key=lambda x: x[1], reverse=True)
   # 가장 유사항 10갸의 영화를 받아 온다.
   sim_scores = sim_scores[1:11]
   # 가장 유사한 10개의 인데스를 가져온다.
   movies_indices = [idx[0] for idx in sim_scores]
   # 가장 유사한 10개의 영화의 제목을 리턴한다.
   return data['title'].iloc[movies_indices]

In [45]:
get_recommendation("Toy Story")

15348               Toy Story 3
2997                Toy Story 2
10301    The 40 Year Old Virgin
8327                  The Champ
1071      Rebel Without a Cause
11399    For Your Consideration
1932                  Condorman
3057            Man on the Moon
485                      Malice
11606              Factory Girl
Name: title, dtype: str